# 6. Predictive Analysis — Classification

Trains and compares four classifiers (Logistic Regression, SVM, Decision Tree, KNN)
to predict Falcon 9 first-stage landing outcome from pre-launch features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

data = pd.read_csv('../data/spacex_launch_data.csv')
Y = data['Class'].to_numpy()

X = pd.read_csv('../data/spacex_features_one_hot.csv')
print("Feature matrix shape:", X.shape)

Feature matrix shape: (90, 83)


`X` has 83 columns: 5 numeric features unchanged (FlightNumber, PayloadMass,
Flights, Block, ReusedCount) plus one-hot-encoded Orbit (11), LaunchSite (3),
LandingPad (5), Serial (58), and the three boolean flags GridFins/Reused/Legs
(2 each).

In [ ]:
transform = StandardScaler()
X_scaled = transform.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=2)
print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])

Train size: 72  Test size: 18


## 6.1 Logistic Regression

In [ ]:
parameters = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr_cv = GridSearchCV(LogisticRegression(), parameters, cv=10)
lr_cv.fit(X_train, Y_train)
print("Best params:", lr_cv.best_params_)
print("Validation accuracy:", lr_cv.best_score_)
print("Test accuracy:", lr_cv.score(X_test, Y_test))

## 6.2 Support Vector Machine

In [ ]:
parameters = {
    'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
    'C': np.logspace(-3, 3, 5),
    'gamma': np.logspace(-3, 3, 5),
}
svm_cv = GridSearchCV(SVC(), parameters, cv=10)
svm_cv.fit(X_train, Y_train)
print("Best params:", svm_cv.best_params_)
print("Validation accuracy:", svm_cv.best_score_)
print("Test accuracy:", svm_cv.score(X_test, Y_test))

Best params: {'C': 1.0, 'gamma': 0.0316, 'kernel': 'sigmoid'}
Validation accuracy: 0.8196
Test accuracy: 0.8333


## 6.3 Decision Tree

In [ ]:
parameters = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [2 * n for n in range(1, 10)],
    'max_features': ['sqrt'],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
}
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=2), parameters, cv=10)
tree_cv.fit(X_train, Y_train)
print("Best params:", tree_cv.best_params_)
print("Validation accuracy:", tree_cv.best_score_)
print("Test accuracy:", tree_cv.score(X_test, Y_test))

Best params: {'criterion': 'gini', 'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'splitter': 'best'}
Validation accuracy: 0.8625
Test accuracy: 0.8333


> **Reproducibility note:** `DecisionTreeClassifier` has internal randomness
> (tie-breaking between equally good splits) that isn't fully pinned down even
> with `random_state` set on some scikit-learn versions when combined with
> `GridSearchCV`'s own refitting. Re-running this cell can shift test accuracy
> by a few percentage points — treat the *validation* accuracy from cross-validation
> as the more stable number for model comparison, and always report the *exact*
> value your own run produces.

## 6.4 K-Nearest Neighbors

In [ ]:
parameters = {
    'n_neighbors': list(range(1, 11)),
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2],
}
knn_cv = GridSearchCV(KNeighborsClassifier(), parameters, cv=10)
knn_cv.fit(X_train, Y_train)
print("Best params:", knn_cv.best_params_)
print("Validation accuracy:", knn_cv.best_score_)
print("Test accuracy:", knn_cv.score(X_test, Y_test))

## 6.5 Model comparison & confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'],
    'Validation Accuracy': [lr_cv.best_score_, svm_cv.best_score_, tree_cv.best_score_, knn_cv.best_score_],
    'Test Accuracy': [lr_cv.score(X_test, Y_test), svm_cv.score(X_test, Y_test),
                       tree_cv.score(X_test, Y_test), knn_cv.score(X_test, Y_test)],
})
print(results)

best_model = tree_cv  # highest validation accuracy in this run
yhat = best_model.predict(X_test)
cm = confusion_matrix(Y_test, yhat)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not land', 'Landed'], yticklabels=['Did not land', 'Landed'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Decision Tree')
plt.show()

**Conclusion:** The Decision Tree classifier had the highest 10-fold cross-validation
accuracy (86.3%) among the four models and matched the others on the held-out test
set (83.3%, i.e. 15 of 18 correct). Given the small test set (18 samples), the
cross-validation score is the more reliable basis for model selection.